In [0]:
from pyspark.sql.functions import *

In [0]:
# =========================================================
# 1. SLV MONTH temp 로드
# =========================================================
fact_df = spark.table(
    "hive_metastore.demo_airstatus_silver.SLV_temp_fact_air_quality_month"
)

In [0]:
# =========================================================
# 2. 관측소 메타 (dmX / dmY 타입 강제 일치)
# =========================================================
stations_df = spark.table(
    "hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations"
).select(
    "stationName",
    col("dmX").cast("double").alias("dmX"),   # ✅ 핵심
    col("dmY").cast("double").alias("dmY"),   # ✅ 핵심
    "addr"
)

In [0]:

# =========================================================
# 3. Geo 정보 조인
# =========================================================
fact_geo_df = (
    fact_df
    .join(
        stations_df,
        on="stationName",
        how="left"
    )
)

In [0]:
display(fact_geo_df[fact_geo_df["stationName"] =="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value,year,month,day,hour,dmX,dmY,addr
중구,2026-05-06T08:00:00Z,72,2,21,11,2026,5,6,17,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-06T23:00:00Z,59,2,34,20,2026,5,7,8,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-04T16:00:00Z,63,2,25,9,2026,5,5,1,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-07T05:00:00Z,60,2,17,11,2026,5,7,14,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-06T21:00:00Z,59,2,28,18,2026,5,7,6,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-05T03:00:00Z,66,2,20,10,2026,5,5,12,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-05T21:00:00Z,77,2,23,9,2026,5,6,6,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-03T23:00:00Z,66,2,25,7,2026,5,4,8,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-07T13:00:00Z,48,1,19,9,2026,5,7,22,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동
중구,2026-05-04T02:00:00Z,68,2,28,12,2026,5,4,11,37.564639,126.975961,서울 중구 덕수궁길 15 시청서소문별관 3동


In [0]:
# =========================================================
# 4. Gold 테이블 append (기존 데이터 유지)
# =========================================================
fact_geo_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_gold.GLD_fact_air_quality_dashboard"
    )

print("✅ GLD_MONTH data appended successfully.")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5386308739448738>, line 7
      1 # =========================================================
      2 # 4. Gold 테이블 append (기존 데이터 유지)
      3 # =========================================================
      4 fact_geo_df.write \
      5     .mode("append") \
      6     .format("delta") \
----> 7     .saveAsTable(
      8         "hive_metastore.demo_airstatus_gold.GLD_fact_air_quality_dashboard"
      9     )
     11 print("✅ GLD_MONTH data appended successfully.")

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wrapper,
    220         module_name=module_name,
   (...)
  